# Mini-agent batch + ingest

CPU runtime. Requires `serve_qwen_colab.ipynb` running on A100.

Set `MODEL_API_BASE` from the serve notebook, then run all cells.

In [ ]:
MODEL_API_BASE = 'https://YOUR-TUNNEL.trycloudflare.com/v1'
MODEL_API_KEY = 'EMPTY'
MODEL_NAME = 'Qwen3-8B'
SMOKE_ONLY = False
MAX_STEPS = 25

In [ ]:
import os
if not os.path.isdir('/content/latent_failiure_prediction'):
    !git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
%cd /content/latent_failiure_prediction/stage2
!pip install -q -e ../stage1 -e '.[devbugs]'

In [ ]:
import subprocess, sys
from datetime import datetime

os.environ.update(MODEL_API_BASE=MODEL_API_BASE, MODEL_API_KEY=MODEL_API_KEY, MODEL_NAME=MODEL_NAME)
OUTPUT_DIR = f'data/trajectories/devbugs_run_{datetime.now():%Y%m%d_%H%M%S}'
cmd = [sys.executable, '-m', 'stage2.devbugs.run_batch', '--output-dir', OUTPUT_DIR,
       '--api-base', MODEL_API_BASE, '--api-key', MODEL_API_KEY, '--model', MODEL_NAME,
       '--max-steps', str(MAX_STEPS), '--skip-preflight']
cmd += (['--instance-id', 'mini_add_001'] if SMOKE_ONLY else ['--instances', 'config/devbugs_instances.txt'])
%cd /content/latent_failiure_prediction/stage2
subprocess.run(cmd, check=True)
print('OUTPUT_DIR=', OUTPUT_DIR)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'stage2.trajectories.ingest_batch',
    '--traj-dir', OUTPUT_DIR, '--output-dir', 'data/normalized'], check=True)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/normalized', 'zip', 'data/normalized')
files.download('/content/normalized.zip')